# 03 - Prediction
### DI Leather · Forecasting weekly production output (`actual_qty`)

| | |
|---|---|
| **Input** | `production_model.joblib` (from Notebook 2) and a table describing the new production order(s) |
| **Output** | Predicted number of shoes that will actually be produced |

Use this notebook to forecast output for a new order **before** production starts.

---
## Step 1 - Imports

In [1]:
import pandas as pd
import joblib

## Step 2 - Prediction function
The function does three things:
1. **Loads** the saved pipeline.
2. **Repeats the same feature engineering** as Notebook 1 - the three new ratio columns must be created in exactly the same way, because the model was trained with them.
3. **Predicts** and rounds to whole shoes.

The input can contain several rows; one prediction is returned per row.

In [2]:
def predict_production(new_data, model_path='production_model.joblib'):
    """
    Loads the saved model and predicts actual_qty for new data.
    new_data: A pandas DataFrame containing the pre-production features.
    """
    # 1. Load the saved pipeline
    pipeline = joblib.load(model_path)

    # 2. Apply the exact same feature engineering
    # .copy() avoids modifying the original input dataframe
    df_pred = new_data.copy()
    df_pred['theoretical_max_production'] = (df_pred['working_hours'] * 60) / df_pred['total_duration_per_shoe_mins']
    df_pred['machine_worker_ratio'] = df_pred['machine_count'] / df_pred['no_of_workers']
    df_pred['avg_hours_per_worker'] = df_pred['working_hours'] / df_pred['no_of_workers']

    # 3. Predict
    predictions = pipeline.predict(df_pred)

    # Return whole numbers, since the output is a quantity of shoes
    return predictions.round().astype(int)

## Step 3 - Describe a new production order
The input must contain these **14 columns**:

| Column | Meaning | Example |
|---|---|---|
| `financial_month` | Month name (Apr … Mar) | `'Apr'` |
| `week_no` | Week number of the year | `4` |
| `week_of_month` | Week within the month (1–5) | `4` |
| `factory_unit` | Factory unit | `'Factory Unit 01'` |
| `product_family` | Ladies / Gents | `'Ladies'` |
| `item_name` | Item name | `'BINOLI (VAST) - BROWN'` |
| `size_id` | Shoe size | `10` |
| `grade` | A / B | `'A'` |
| `rate` | Rate value | `4500.00` |
| `no_of_workers` | Workers assigned | `18` |
| `working_hours` | Total working hours | `800.5` |
| `total_duration_per_shoe_mins` | Standard minutes per shoe | `120.0` |
| `machine_count` | Machines available | `5` |
| `order_category` | School / Forces/Police / Others | `'Others'` |

**Not needed:** `planned_qty`, `production_gap_qty`, `bottleneck_flag` and the dates - the model does not use them.

**Unseen values:** an item the model has never seen is estimated using the overall average output; an unseen category in the other text columns is ignored.

In [3]:
new_order = pd.DataFrame([{
    'financial_month': 'Apr',
    'week_no': 4,
    'week_of_month': 4,
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Ladies',
    'item_name': 'BINOLI (VAST) - BROWN',
    'size_id': 10,
    'grade': 'A',
    'rate': 4500.00,
    'no_of_workers': 18,
    'working_hours': 800.5,
    'total_duration_per_shoe_mins': 120.0,
    'machine_count': 5,
    'order_category': 'Others'
}])

new_order

,financial_month,week_no,week_of_month,factory_unit,product_family,item_name,size_id,grade,rate,no_of_workers,working_hours,total_duration_per_shoe_mins,machine_count,order_category
0,Apr,4,4,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,10,A,4500.0,18,800.5,120.0,5,Others


## Step 4 - Get the prediction

In [4]:
predicted_qty = predict_production(new_order)

print(f"Predicted Actual Production Qty: {predicted_qty[0]} shoes")

Predicted Actual Production Qty: 43 shoes


### Reading the output
The model forecasts that this order will actually produce about **43 shoes**. Based on the test results in Notebook 2, expect a typical error of roughly **±3 - 4 shoes** (test MAE 3.41, RMSE 4.35). Treat the number as a planning estimate, not a guarantee.

Comparing this forecast with the quantity planned for the order shows the expected shortfall or surplus: `planned quantity − predicted quantity`.
